In [1]:
from chggen.common.sample_utils import CSP_Generator
from chggen.common.data_utils import mkdir
from chggen.common.sample_utils import get_inpaint_data_fromHost
from chggen.common.sample_utils import get_batch_inpaint_data_fromHost
from chggen.common.sample_utils import get_coarse_grain_framework, filter_nan_structure, compute_ewald_energy_single_structure
from chggen.common.sample_utils import run_SDE_simpleCubic
from chggen.common.e_hull_calculator import EHullCalculator

from types import SimpleNamespace
import numpy as np

from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core import Structure, Composition, Element, Lattice
from pymatgen.io.cif import CifWriter

import argparse
import pandas as pd
import os
import time
from datetime import datetime

def relax_structures(csp, s_list):
    s_list_relax = []
    for s in s_list:
        atoms = AseAtomsAdaptor().get_atoms(s)

        result = csp.relaxer.relax(atoms= atoms,
                            fmax = 0.1,
                            steps = 2000,
                            relax_cell = True,
                            verbose = True,
                            # trajectory_path = None,
        )
        s_relax = result["final_structure"]
        s_list_relax.append(s_relax)

    return s_list_relax

def coarse_grain_framework(s_list, species_to_remove):
    
    host_structure_list = []
    num_intercalat_list = []

    for s in s_list:
        analyzer_asGen = SpacegroupAnalyzer(structure= s, symprec= 0.15, angle_tolerance= 15)
        symbol_asGen = analyzer_asGen.get_space_group_symbol()
        print("As generated spacegroup: ", symbol_asGen)

        s_frame, symbol_frame, num_species = get_coarse_grain_framework(s, species_to_remove= species_to_remove)
        s_frame = s_frame.get_primitive_structure()

        if symbol_frame in ['P1', 'P-1', 'Pm']: 
            continue

        host_structure_list.append(s_frame)
        num_intercalat_list.append(int(num_species))

    return host_structure_list, num_intercalat_list



def refine_structure(s_list, symprec= 0.2, angle_tolerance= 15):
    s_list_refine = []

    for s in s_list:
        analyzer_inpaint = SpacegroupAnalyzer(structure= s, symprec= symprec, angle_tolerance= angle_tolerance)
        try:
            symbol_inpaint = analyzer_inpaint.get_space_group_symbol()
        except:
            print("Failed to get space group symbol")
            continue

        if symbol_inpaint== 'P1' or symbol_inpaint== 'P-1':
            continue
        else:
            print(symbol_inpaint)
            s_conventional_unit = analyzer_inpaint.get_conventional_standard_structure()
            s_list_refine.append(s_conventional_unit)

    return s_list_refine


# %%
def get_save_dict_list(csp, s_list, type):
    save_dict_list = []
    for ii, s in enumerate(s_list):
        atoms = AseAtomsAdaptor().get_atoms(s)

        analyzer_inpaint = SpacegroupAnalyzer(structure= s, symprec= 0.15, angle_tolerance= 15)
        try:
            symbol_inpaint = analyzer_inpaint.get_space_group_symbol()
        except:
            print("Failed to get space group symbol")
            continue
        
        prediction = csp.chgnet.predict_structure(s)
        E0_atom = prediction['e'] 
        F_max = np.max(np.abs(prediction['f']))


        result = csp.relaxer.relax(atoms= atoms,
                            fmax = 0.1,
                            steps = 2000,
                            relax_cell = True,
                            verbose = True,
                            # trajectory_path = None,
        )
        s_relax = result["final_structure"]
        toten = result['trajectory'].energies[-1]

        analyzer_relax = SpacegroupAnalyzer(structure= s_relax, symprec= 0.15, angle_tolerance= 15)
        symbol_relax = analyzer_relax.get_space_group_symbol()
        s_relax_refine = analyzer_relax.get_refined_structure()

        if symbol_relax == 'P1' or symbol_relax == 'P-1':
            continue

        save_dict ={'material_id': hex(int(time.time()*1e8)),
                    'formula': s_relax.composition.reduced_formula,
                    's_asGen_cif': str(CifWriter(s)),
                    'spacegroup_asGen': symbol_inpaint,
                    's_relax_refine_cif': str(CifWriter(s_relax_refine)),
                    'spacegroup_refine': symbol_relax,
                    's_relax_cif': str(CifWriter(s_relax)),
                    'Fmax_chgnet': F_max, 
                    'E0_chgnet_atom': E0_atom,
                    'E_chgnet_atom': toten / s_relax.num_sites,
                    'type': type,
                    'energy': toten,
                    'structure': s_relax,
                    # 'lattice_type': type_init,
                    # 'mutation': mutation
                    }
        save_dict_list.append(save_dict)
    return save_dict_list



/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
comp_str = "LiPOF4"
### Define the kwargs for the generation ###
ld_kwargs = SimpleNamespace(
        n_step_each = 5,            # Corrector
        min_sigma = 0.01,
        num_noise_level = 200,
        signal_to_noise_ratio = 0.4,
        save_traj = False,
        disable_bar = False,
    )
gen_kwargs = SimpleNamespace(
        num_gen = 10, # number of structures generated from the cubic lattice (not used)
        num_mutation = 2, # number of mutations during the relax-generation iteration (not used)
        num_cell = 20, # number of times to the formula
        ehull_cutoff = 0.06,
        )

### Define the CSP file and patched_phase diagram ###
csp = CSP_Generator(chggen_path = "./files/cut_7_conv_3_epoch=27-val_loss=0.87.ckpt",
                    device='cuda:3')
# csp.load_e_hull_calculator(ppd_path= "/home/zhongpc/chggen/host_gen/file_trans/2023-02-07-ppd-mp.pkl.gz")

s_list = csp.generate_simple_cubic_structure(comp_str = comp_str, # the string of composition,
                                        atom_volume = 12, # avg atom volume,
                                        gen_kwargs = gen_kwargs, # generation keyword arguments,
                                        ld_kwargs = ld_kwargs, # SDE simulation keyword arguments,
                                        ) # chggen

print("Done")
    
# Example: python genFlow.py -d cuda:6 -s Li -v 22 -n 3


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


CHGNet initialized with 412,525 parameters
CHGNet will run on cuda:3
tensor([[12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058],
        [12.8058, 12.8058, 12.8058]], device='cuda:3')


100%|██████████| 199/199 [03:30<00:00,  1.06s/it]


Done


In [4]:
! mkdir ./files/amorphous_LiPOF4/

In [7]:
for ii, structure in enumerate(s_list):
    structure.to(filename= f"./files/amorphous_LiPOF4/s_"+str(ii)+".cif")